In [1]:
#Importing Libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
!pip install mlflow

In [3]:
import mlflow

In [4]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

In [5]:
mlflow.set_experiment("HousePricePrediction")

2026/05/26 08:41:43 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/26 08:41:43 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
2026/05/26 08:41:43 INFO mlflow.tracking.fluent: Experiment with name 'HousePricePrediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/mlops_sample/mlruns/1', creation_time=1779784903294, experiment_id='1', last_update_time=1779784903294, lifecycle_stage='active', name='HousePricePrediction', tags={}>

In [6]:
data = pd.read_csv('House Price Prediction Dataset.csv')

In [7]:
data.head()

,Id,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage,Price
0,1,1360,5,4,3,1970,Downtown,Excellent,No,149919
1,2,4272,5,4,3,1958,Downtown,Excellent,No,424998
2,3,3592,2,2,3,1938,Downtown,Good,No,266746
3,4,966,4,2,2,1902,Suburban,Fair,Yes,244020
4,5,4926,1,4,2,1975,Downtown,Fair,Yes,636056


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Id         2000 non-null   int64 
 1   Area       2000 non-null   int64 
 2   Bedrooms   2000 non-null   int64 
 3   Bathrooms  2000 non-null   int64 
 4   Floors     2000 non-null   int64 
 5   YearBuilt  2000 non-null   int64 
 6   Location   2000 non-null   object
 7   Condition  2000 non-null   object
 8   Garage     2000 non-null   object
 9   Price      2000 non-null   int64 
dtypes: int64(7), object(3)
memory usage: 156.4+ KB


In [9]:
data.describe()

,Id,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Price
count,2000.000000,2000.000000,2000.000000,2000.00000,2000.000000,2000.000000,2000.000000
mean,1000.500000,2786.209500,3.003500,2.55250,1.993500,1961.446000,537676.855000
std,577.494589,1295.146799,1.424606,1.10899,0.809188,35.926695,276428.845719
min,1.000000,501.000000,1.000000,1.00000,1.000000,1900.000000,50005.000000
25%,500.750000,1653.000000,2.000000,2.00000,1.000000,1930.000000,300098.000000
50%,1000.500000,2833.000000,3.000000,3.00000,2.000000,1961.000000,539254.000000
75%,1500.250000,3887.500000,4.000000,4.00000,3.000000,1993.000000,780086.000000
max,2000.000000,4999.000000,5.000000,4.00000,3.000000,2023.000000,999656.000000


In [10]:
data['Location'].unique()

array(['Downtown', 'Suburban', 'Urban', 'Rural'], dtype=object)

In [11]:
data['Condition'].unique()

array(['Excellent', 'Good', 'Fair', 'Poor'], dtype=object)

In [12]:
data['Garage'].unique()

array(['No', 'Yes'], dtype=object)

In [13]:
final_df  = pd.get_dummies(data = data,drop_first=True, columns=['Location', 'Condition', 'Garage'], dtype='int')

In [14]:
final_df.head()

,Id,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Price,Location_Rural,Location_Suburban,Location_Urban,Condition_Fair,Condition_Good,Condition_Poor,Garage_Yes
0,1,1360,5,4,3,1970,149919,0,0,0,0,0,0,0
1,2,4272,5,4,3,1958,424998,0,0,0,0,0,0,0
2,3,3592,2,2,3,1938,266746,0,0,0,0,1,0,0
3,4,966,4,2,2,1902,244020,0,1,0,1,0,0,1
4,5,4926,1,4,2,1975,636056,0,0,0,1,0,0,1


In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score

In [16]:
features = final_df[final_df.columns.drop(['Id','Price', 'YearBuilt'])].values

In [17]:
target = final_df['Price'].values

In [18]:
features

array([[1360,    5,    4, ...,    0,    0,    0],
       [4272,    5,    4, ...,    0,    0,    0],
       [3592,    2,    2, ...,    1,    0,    0],
       ...,
       [1062,    5,    1, ...,    0,    1,    0],
       [4062,    3,    1, ...,    0,    0,    1],
       [2989,    5,    1, ...,    0,    0,    0]])

In [19]:
target

array([149919, 424998, 266746, ..., 476925, 161119, 482525])

In [20]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.20, random_state=42)

In [21]:
training_data_path = final_df.to_csv('cleaned_data.csv')

# Running Different Models

In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [24]:
##  Multiple Linear Regression

In [25]:
with mlflow.start_run():
    mlflow.set_tag("developer", "Praveen")
    mlflow.log_param("data", "/workspaces/mlops_sample/cleaned_data.csv")
    linear_reg = LinearRegression()
    linear_reg.fit(X_train, y_train)
    y_prediction = linear_reg.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_true=y_test, y_pred=y_prediction))
    r2_score = r2_score(y_true=y_test, y_pred=y_prediction)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2_score", r2_score)

In [26]:
with mlflow.start_run(run_name="Polynomial Regression"):
    mlflow.set_tag("developer", "Praveen")
    mlflow.log_param("data", "/workspaces/mlops_sample/cleaned_data.csv")
    mlflow.log_param("algorithm", "Polynomial Function")
    mlflow.log_param("polynomial_degree", "4")
    poly = PolynomialFeatures(degree=4, include_bias=False)
    X_poly = poly.fit_transform(X_train)
    model = LinearRegression()
    model.fit(X_poly, y_train)
    y_prediction = model.predict(poly.fit_transform(X_test))
    rmse = np.sqrt(mean_squared_error(y_true=y_test, y_pred=y_prediction))
    mlflow.log_metric("rmse", rmse)

In [27]:
!pip install xgboost

In [28]:
from xgboost import XGBRegressor

In [29]:
xgb = XGBRegressor()

In [30]:
with mlflow.start_run(run_name="XGB Regressor default"):
    mlflow.set_tag("developer", "Praveen")
    mlflow.log_param("algorithm", "XGB Regressor with default Params")
    mlflow.log_param("data", "/workspaces/mlops_sample/cleaned_data.csv")
    xgb.fit(X_train, y_train)
    y_prediction = xgb.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_true=y_test, y_pred=y_prediction))
    mlflow.log_metric("rmse", rmse)

In [34]:
!pip install hyperopt

     |████████████████████████████████| 1.6 MB 12.8 MB/s eta 0:00:01
     |████████████████████████████████| 203 kB 80.7 MB/s eta 0:00:01


In [35]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

In [38]:
k = 5
print(f'thenumber is {k}')

thenumber is 5


In [36]:
import xgboost as xgb

In [41]:
# 1. Define the search space
space = {
    'max_depth': hp.quniform('max_depth', 3, 10, 1),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'n_estimators': hp.quniform('n_estimators', 100, 1000, 10),
    'reg_alpha': hp.uniform('reg_alpha', 0, 1),
    'reg_lambda': hp.uniform('reg_lambda', 0, 1)
}


# 2. Define the objective function

def objective(params):
    # Convert parameters to correct types (Hyperopt often returns floats)
    params['max_depth'] = int(params['max_depth'])
    params['n_estimators'] = int(params['n_estimators'])
    
    with mlflow.start_run(nested=True):
        mlflow.set_tag("model", "xgboost")
        model = xgb.XGBRegressor(**params, early_stopping_rounds=10, random_state=42)
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
        preds = model.predict(X_test)
        rmse = mean_squared_error(y_test, preds, squared=False)
        actual_trees = model.best_iteration if hasattr(model, 'best_iteration') else params['n_estimators']

# Log to MLflow
        mlflow.log_params(params)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("best_iteration", actual_trees)
        
        return {'loss': rmse, 'status': STATUS_OK}


with mlflow.start_run(run_name="XGB_Hyperopt_Optimization"):
    best_params = fmin(fn = objective, space = space, algo = tpe.suggest, max_evals = 50, trials = Trials())
    mlflow.log_params(best_params)

[0]	validation_0-rmse:282212.39253                                                                                         
[1]	validation_0-rmse:283091.62061                                                                                         
[2]	validation_0-rmse:283589.50348                                                                                         
[3]	validation_0-rmse:285447.53838                                                                                         
[4]	validation_0-rmse:285923.36144                                                                                         
[5]	validation_0-rmse:286095.05519                                                                                         
[6]	validation_0-rmse:288719.90786                                                                                         
[7]	validation_0-rmse:291126.70455                                                                                         
[8]	vali

[2]	validation_0-rmse:282111.34803                                                                                         
[3]	validation_0-rmse:282430.82342                                                                                         
[4]	validation_0-rmse:284363.26810                                                                                         
[5]	validation_0-rmse:285414.37229                                                                                         
[6]	validation_0-rmse:286288.93840                                                                                         
[7]	validation_0-rmse:286703.56534                                                                                         
[8]	validation_0-rmse:288122.36662                                                                                         
[9]	validation_0-rmse:288721.75457                                                                                         
[10]	val

[5]	validation_0-rmse:280211.50652                                                                                         
[6]	validation_0-rmse:280393.86904                                                                                         
[7]	validation_0-rmse:280441.62603                                                                                         
[8]	validation_0-rmse:280653.90536                                                                                         
[9]	validation_0-rmse:280907.10562                                                                                         
[0]	validation_0-rmse:279712.63725                                                                                         
[1]	validation_0-rmse:280178.16651                                                                                         
[2]	validation_0-rmse:280425.32113                                                                                         
[3]	vali

[7]	validation_0-rmse:282565.58688                                                                                         
[8]	validation_0-rmse:282839.09851                                                                                         
[9]	validation_0-rmse:282946.17028                                                                                         
[10]	validation_0-rmse:283377.92865                                                                                        
[0]	validation_0-rmse:297851.08288                                                                                         
[1]	validation_0-rmse:314863.66652                                                                                         
[2]	validation_0-rmse:322688.03861                                                                                         
[3]	validation_0-rmse:325715.13028                                                                                         
[4]	vali

[10]	validation_0-rmse:281370.29539                                                                                        
[0]	validation_0-rmse:278973.99546                                                                                         
[1]	validation_0-rmse:279049.56903                                                                                         
[2]	validation_0-rmse:279138.58903                                                                                         
[3]	validation_0-rmse:279254.19986                                                                                         
[4]	validation_0-rmse:279354.07650                                                                                         
[5]	validation_0-rmse:279486.81897                                                                                         
[6]	validation_0-rmse:279612.94913                                                                                         
[7]	vali

[2]	validation_0-rmse:279782.94238                                                                                         
[3]	validation_0-rmse:279945.44056                                                                                         
[4]	validation_0-rmse:280099.77267                                                                                         
[5]	validation_0-rmse:280311.15378                                                                                         
[6]	validation_0-rmse:280771.34757                                                                                         
[7]	validation_0-rmse:280640.31886                                                                                         
[8]	validation_0-rmse:280709.40939                                                                                         
[9]	validation_0-rmse:280803.53432                                                                                         
[10]	val

[4]	validation_0-rmse:304311.98889                                                                                         
[5]	validation_0-rmse:303256.63884                                                                                         
[6]	validation_0-rmse:304302.42907                                                                                         
[7]	validation_0-rmse:308332.01066                                                                                         
[8]	validation_0-rmse:312087.57315                                                                                         
[9]	validation_0-rmse:318082.46481                                                                                         
[10]	validation_0-rmse:322111.75138                                                                                        
[0]	validation_0-rmse:280020.37258                                                                                         
[1]	vali

[6]	validation_0-rmse:282098.18000                                                                                         
[7]	validation_0-rmse:282567.69298                                                                                         
[8]	validation_0-rmse:283034.14335                                                                                         
[9]	validation_0-rmse:283420.57714                                                                                         
[10]	validation_0-rmse:283873.92229                                                                                        
[0]	validation_0-rmse:279045.16424                                                                                         
[1]	validation_0-rmse:279504.00590                                                                                         
[2]	validation_0-rmse:280012.14577                                                                                         
[3]	vali

[10]	validation_0-rmse:290545.43417                                                                                        
100%|█████████████████████████████████████████████████████| 50/50 [00:13<00:00,  3.72trial/s, best loss: 278940.6270832908]


In [43]:
mlflow.autolog()

with mlflow.start_run(run_name="XGB Regressor autolog"):
    mlflow.set_tag("developer", "Praveen")
    mlflow.log_param("algorithm", "XGB Regressor with Autolog Params")
    mlflow.log_param("data", "/workspaces/mlops_sample/cleaned_data.csv")
    autolog_model = xgb.XGBRegressor(n_estimators = 810, learning_rate = 0.11014099849219953, max_depth = 4, reg_alpha = 0.9493672374210916, reg_lambda = 0.6839771610567229)
    autolog_model.fit(X_train, y_train)
    y_prediction = autolog_model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_true=y_test, y_pred=y_prediction))
    mlflow.log_metric("rmse", rmse)

2026/05/26 10:50:51 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.3.0 <= scikit-learn <= 1.7.0, but the installed version is 1.0.2. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2026/05/26 10:50:52 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/05/26 10:50:52 WARNING mlflow.utils.autologging_utils: MLflow statsmodels autologging is known to be compatible with 0.14.1 <= statsmodels <= 0.14.4, but the installed version is 0.13.2. If you encounter errors during autologging, try upgrading / downgrading statsmodels to a compatible version, or try upgrading MLflow.
2026/05/26 10:50:52 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.
2026/05/26 10:50:52 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/05/26 10:50:53 WARNING mlflow.models.model: `artifact_path`